In [1]:
# 本部分用于评价LLM的应用程序

In [2]:
import os

from Scripts.pywin32_postinstall import verbose
from langchain_openai import OpenAIEmbeddings

api_key = os.environ.get("DEEPSEEK_API_KEY")

In [26]:
from langchain_classic.chains import RetrievalQA
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.document_loaders import CSVLoader
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_classic.vectorstores import DocArrayInMemorySearch
from langchain_openai.embeddings import OpenAIEmbeddings

In [15]:
file = "OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file, encoding="utf-8") # 注意说明编码
data = loader.load()

In [29]:
# 创建embedding对象
embeddings = OpenAIEmbeddings(
    api_key= os.environ.get("QWEN_API_KEY"), # 填写千问的api-key
    # url
    base_url= "https://dashscope.aliyuncs.com/compatible-mode/v1",
    # 填写目标模型
    model = "qwen3.7-text-embedding",
    # 发送原始文本
    check_embedding_ctx_length=False,
    # 根据DashScope的文档，在此处限制单词发送的大小为20
    chunk_size=20
)

# 创建索引
index = VectorstoreIndexCreator(
    # 需要填写embedding参数，未使用OpenAI官方的embedding模型需要手动导入
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings
).from_loaders([loader])

In [33]:
# 设置语言模型
llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash",
    temperature=0.0
)


qa = RetrievalQA.from_chain_type(
    llm=llm,
    # 设置链的类型
    chain_type="stuff",
    # 设置检索器
    retriever=index.vectorstore.as_retriever(),
    verbose=True, # 设置打印日志详细程度
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>"
    }
)

In [1]:
# 自身预设一些设置好的数据集
data[10]

NameError: name 'data' is not defined